## Setup: Import Dependencies

Importing PySpark SQL functions and Window for aggregate calculations and rolling window operations.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

## Configuration: Catalog and Table Paths

Defining the catalog structure:
* **Catalog**: `chicago_city`
* **Source**: Silver layer (`silver.silver_trips`)
* **Destination**: Gold layer (`gold.*`) — business-ready aggregates

In [0]:
CATALOG = "chicago_city"
SCHEMA="gold"
SILVER_TABLE="silver_trips"
GOLD_TABLE="gold_trips"

SILVER_PATH=f"{CATALOG}.silver.{SILVER_TABLE}"

GOLD_PATH=f"{CATALOG}.{SCHEMA}.{GOLD_TABLE}"


## Load Silver Data

Loading the cleaned and enriched trip data from the Silver layer. This dataset will be used for all Gold aggregates.

In [0]:
dfsilver = spark.table(SILVER_PATH)
# dfsilver.cache() # Gold reads Silver multiple times — cache saves recomputation

print(f"Silver rows available for Gold: {dfsilver.count():,}")

Silver rows available for Gold: 1,799,650


## Gold Aggregate 1: Revenue by Community Area

**Business Question**: Which pickup zones generate the most revenue?

**Metrics**:
* Trip count
* Total and average fare
* Total and average trip miles

Grouped by pickup community area and month, ordered by total revenue.

In [0]:
# ── Aggregate 1: Revenue by community area ───────────────────────────────────
# Business question: which pickup zones generate the most revenue?

df_revenue_by_area = (
    dfsilver
    .groupBy("pickup_community_area", "year_month")
    .agg(
        F.count("trip_id").alias("trip_count"),
        F.round(F.sum("fare"), 2).alias("total_fare"),
        F.round(F.avg("fare"), 2).alias("avg_fare"),
        F.round(F.sum("trip_miles"), 2).alias("total_miles"),
        F.round(F.avg("trip_miles"), 2).alias("avg_miles"),
    )
    .filter(F.col("pickup_community_area").isNotNull())
    .orderBy("year_month", F.desc("total_fare"))
)

GOLD_REVENUE_TABLE = f"{CATALOG}.{SCHEMA}.revenue_by_area"

(
    df_revenue_by_area.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_REVENUE_TABLE)
)


## Optimize Revenue Table

Applying **ZORDER** on `pickup_community_area` to co-locate related data on disk. This significantly speeds up BI queries that filter by community area.

In [0]:
# ZORDER on the most-queried column — co-locates data on disk for faster BI queries
spark.sql(f"OPTIMIZE {GOLD_REVENUE_TABLE} ZORDER BY (pickup_community_area)")

print("Gold aggregate 1: revenue_by_area — done")
df_revenue_by_area.show(10)

Gold aggregate 1: revenue_by_area — done
+---------------------+----------+----------+----------+--------+-----------+---------+
|pickup_community_area|year_month|trip_count|total_fare|avg_fare|total_miles|avg_miles|
+---------------------+----------+----------+----------+--------+-----------+---------+
|                 76.0|   2023-01|     63319|2447506.45|   38.65|  883499.24|    13.95|
|                  8.0|   2023-01|     76829| 991713.38|   12.91|  256900.89|     3.34|
|                 32.0|   2023-01|     55592| 699581.73|   12.58|  184712.92|     3.32|
|                 28.0|   2023-01|     37443| 500627.27|   13.37|  131314.91|     3.51|
|                 56.0|   2023-01|     11888| 407404.25|   34.27|  139911.36|    11.77|
|                  6.0|   2023-01|     12505| 195183.92|   15.61|   55090.47|     4.41|
|                 33.0|   2023-01|      7756| 143888.22|   18.55|   42742.12|     5.51|
|                  3.0|   2023-01|      6882| 125861.85|   18.29|    36897.2|  

## Gold Aggregate 2: Peak Demand by Hour

**Business Question**: What hours see the highest trip volume?

**Metrics**:
* Trip count by hour of day
* Average fare
* Average trip duration (in minutes)

Useful for capacity planning and surge pricing analysis.

In [0]:
# ── Aggregate 2: Peak demand by hour-of-day ───────────────────────────────────
# Business question: what hours see the highest trip volume?

GOLD_DEMAND_TABLE = f"{CATALOG}.{SCHEMA}.demand_by_hour"

df_demand_by_hour = (
    dfsilver
    .groupBy("pickup_hour", "pickup_date")
    .agg(
        F.count("trip_id").alias("trip_count"),
        F.round(F.avg("fare"), 2).alias("avg_fare"),
        F.round(F.avg("trip_seconds") / 60, 1).alias("avg_duration_mins")
    )
)

(
    df_demand_by_hour.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_DEMAND_TABLE)
)

spark.sql(f"OPTIMIZE {GOLD_DEMAND_TABLE} ZORDER BY (pickup_hour)")

print("Gold aggregate 2: demand_by_hour — done")
spark.table(GOLD_DEMAND_TABLE).orderBy(F.desc("trip_count")).show(10)

Gold aggregate 2: demand_by_hour — done
+-----------+-----------+----------+--------+-----------------+
|pickup_hour|pickup_date|trip_count|avg_fare|avg_duration_mins|
+-----------+-----------+----------+--------+-----------------+
|         17| 2023-04-27|      1627|   22.03|             28.9|
|         17| 2023-04-17|      1581|   21.65|             23.6|
|         18| 2023-02-24|      1569|   17.13|             19.1|
|         17| 2023-03-16|      1556|   19.61|             24.7|
|         16| 2023-04-27|      1553|   22.98|             29.6|
|         16| 2023-02-22|      1547|   18.95|             23.7|
|         18| 2023-03-16|      1540|   19.53|             23.2|
|         16| 2023-04-17|      1519|   21.44|             21.9|
|         17| 2023-02-23|      1515|   18.13|             20.8|
|         17| 2023-02-24|      1504|   18.68|             21.6|
+-----------+-----------+----------+--------+-----------------+
only showing top 10 rows


## Gold Aggregate 3: Rolling 7-Day Fare Trends

**Business Question**: Is average fare trending up or down in each zone?

**Technique**: Window function with 7-day lookback
* Partition by community area
* Order by date
* Calculate rolling average fare and trip volume

This pattern is essential for time-series trend analysis in production data pipelines.

In [0]:
# ── Aggregate 3: 7-day rolling average fare by community area ─────────────────
# Business question: is average fare trending up or down in each zone?
# This uses a window function — the signal that separates senior engineers from juniors.

GOLD_ROLLING_TABLE = f"{CATALOG}.{SCHEMA}.rolling_fare_by_area"

window_spec = (
    Window
    .partitionBy("pickup_community_area")
    .orderBy("pickup_date")
    .rowsBetween(-6, 0)
)

df_daily_area = (
    dfsilver
    .groupBy("pickup_community_area", "pickup_date")
    .agg(
        F.count("trip_id").alias("daily_trips"),
        F.round(F.avg("fare"), 2).alias("daily_avg_fare")
    )
    .filter(F.col("pickup_community_area").isNotNull())
)

df_rolling = (
    df_daily_area
    .withColumn("rolling_7d_avg_fare", F.round(F.avg("daily_avg_fare").over(window_spec), 2))
    .withColumn("rolling_7d_trip_count", F.sum("daily_trips").over(window_spec))
)

(
    df_rolling.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_ROLLING_TABLE)
)

spark.sql(f"OPTIMIZE {GOLD_ROLLING_TABLE} ZORDER BY (pickup_community_area, pickup_date)")

print("Gold aggregate 3: rolling_fare_by_area — done")
spark.table(GOLD_ROLLING_TABLE).show(15)
print("\nAll Gold aggregates complete.")

Gold aggregate 3: rolling_fare_by_area — done
+---------------------+-----------+-----------+--------------+-------------------+---------------------+
|pickup_community_area|pickup_date|daily_trips|daily_avg_fare|rolling_7d_avg_fare|rolling_7d_trip_count|
+---------------------+-----------+-----------+--------------+-------------------+---------------------+
|                  1.0| 2023-01-01|         60|          26.0|               26.0|                   60|
|                  1.0| 2023-01-02|         69|         24.47|              25.24|                  129|
|                  1.0| 2023-01-03|         88|         22.06|              24.18|                  217|
|                  1.0| 2023-01-04|        103|         22.89|              23.86|                  320|
|                  1.0| 2023-01-05|        109|         25.64|              24.21|                  429|
|                  1.0| 2023-01-06|         92|         22.33|               23.9|                  521|
|        